In [ ]:
import SimpleITK as sitk
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from scipy import ndimage
import logging
import datetime

logging.basicConfig(level=logging.DEBUG, 
                    format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# TODO: improve error handling

DISTANCE_THERSHOLD = 2.0 # mm
CONTACT_AREA_THRESHOLD = 10.0 # mm2
INTENSITY_DIFF_THRESHOLD = 0.2 # relative threshold
DILATION_RADIUS = 1 # voxels
LOG_LEVEL_NAME = "INFO" # DEBUG or INFO

if LOG_LEVEL_NAME == "DEBUG":
    DEBUG = True
else:
    DEBUG = False

LOG_LEVEL = getattr(logging, LOG_LEVEL_NAME)

logger.info(f"DISTANCE_THERSHOLD: {DISTANCE_THERSHOLD}")
logger.info(f"CONTACT_AREA_THRESHOLD: {CONTACT_AREA_THRESHOLD}")
logger.info(f"INTENSITY_DIFF_THRESHOLD: {INTENSITY_DIFF_THRESHOLD}")
logger.info(f"DILATION_RADIUS: {DILATION_RADIUS}")

logger.setLevel(LOG_LEVEL) # Not sure why I have to declare the level again for debug msgs to work

# logger.debug("test debug")

In [ ]:
class DataLoader:
    def __init__(self, mri_path, annotation_path):
        self.mri_path = mri_path
        self.annotation_path = annotation_path

        self.mri_image = None
        self.annotation_image = None
        self.spacing = None
        self.num_slides = None
        self.node_labels = None
        self.node_masks = {}
        self.node_stats = {}
        self.adjacency_graph = None

        logger.info("DataLoader initialized")

    def load_data(self):
        """Load MRI and annotation data + some checking."""
        logger.info(f"Loading MRI image from {self.mri_path}")
        self.mri_image = sitk.ReadImage(self.mri_path)
        
        logger.info(f"Loading annotation image from {self.annotation_path}")
        self.annotation_image = sitk.ReadImage(self.annotation_path)

        # Ensure same coordinate system
        if not self.check_coordinate_match():
            logger.warning("MRI and annotation images might not be in the same coordinate system!")
        
        self.spacing = self.mri_image.GetSpacing()
        logger.info(f"Image spacing: {self.spacing}")
        
        # Extract node labels
        np_annotation = sitk.GetArrayFromImage(self.annotation_image)
        self.node_labels = np.unique(np_annotation)
        self.node_labels = self.node_labels[self.node_labels > 0]  # Remove background
        
        logger.info(f"Found {len(self.node_labels)} lymph node annotations with labels: {self.node_labels}")
        
        # Create individual masks for each node
        self.create_node_masks()
        
        return self
        
    def check_coordinate_match(self):
        """Helper for the load_data function"""
        """Check if MRI and annotation images have matching coordinate systems."""
        mri_size = self.mri_image.GetSize()
        anno_size = self.annotation_image.GetSize()
        mri_spacing = self.mri_image.GetSpacing()
        anno_spacing = self.annotation_image.GetSpacing()
        mri_origin = self.mri_image.GetOrigin()
        anno_origin = self.annotation_image.GetOrigin()
        
        size_match = mri_size == anno_size
        spacing_match = all(abs(m - a) < 1e-3 for m, a in zip(mri_spacing, anno_spacing))
        origin_match = all(abs(m - a) < 1e-3 for m, a in zip(mri_origin, anno_origin))

        self.num_slides = anno_size[2]
        
        logger.info(f"Size match: {size_match}, Spacing match: {spacing_match}, Origin match: {origin_match}")
        logger.info(f"MRI spacing: {mri_spacing}, Anno spacing: {anno_spacing}")
        logger.info(f"xyz: {anno_size}, num_slides: {self.num_slides}")
        
        return size_match and spacing_match and origin_match
    
    def create_node_masks(self):
        """Helper for the load_data function"""
        """Create binary masks for each lymph node."""
        for label in self.node_labels:
            logger.info(f"Creating mask for node {label}")
            
            # Create binary mask for this node
            node_mask = sitk.Equal(self.annotation_image, int(label))
            self.node_masks[label] = node_mask
            
            # Calculate basic statistics for this node
            np_mri = sitk.GetArrayFromImage(self.mri_image)
            np_mask = sitk.GetArrayFromImage(node_mask)
            node_voxels = np_mri[np_mask > 0]
            
            if len(node_voxels) > 0:
                self.node_stats[label] = {
                    'mean_intensity': np.mean(node_voxels),
                    'std_intensity': np.std(node_voxels),
                    'volume_mm3': np.sum(np_mask) * np.prod(self.spacing),
                    'voxel_count': np.sum(np_mask)
                }
                logger.info(f"  Node {label} stats: {self.node_stats[label]}")
            else:
                logger.warning(f"  Node {label} has no voxels!")

    def get_slices_with_mask(self, node_label):
        """
        Returns a list of slice IDs where the specified node mask exists.
        
        Args:
            node_label: The label of the node to check
            
        Returns:
            List of slice IDs (z-indices) containing the mask
        """
        if node_label not in self.node_masks:
            logger.error(f"Node label {node_label} not found in node masks!")
            return []
        
        # Convert the SimpleITK mask to a numpy array
        mask_array = sitk.GetArrayFromImage(self.node_masks[node_label]) > 0
        
        # Find slices where the mask has at least one True value
        # The first dimension in the numpy array corresponds to the z-axis (slices)
        slices_with_mask = []
        for slice_id in range(mask_array.shape[0]):
            if np.any(mask_array[slice_id]):
                slices_with_mask.append(slice_id)
        
        logger.debug(f"Node {node_label} appears in {len(slices_with_mask)} slices: {slices_with_mask}")
        
        return slices_with_mask

    def get_common_slices(self, node_a, node_b):
        """
        Returns slice IDs where both node masks exist.
        
        Args:
            node_a: First node label
            node_b: Second node label
            
        Returns:
            List of slice IDs where both masks are present
        """
        slices_a = set(self.get_slices_with_mask(node_a))
        slices_b = set(self.get_slices_with_mask(node_b))
        
        common_slices = sorted(list(slices_a.intersection(slices_b)))
        
        logger.info(f"Nodes {node_a} and {node_b} appear together in {len(common_slices)} slices: {common_slices}")
        
        return common_slices


In [ ]:
class SliceAnalyzer:
    # def __init__(self, node_a, node_b, slice_id):
    #     self.node_a = node_a
    #     self.node_b = node_b
    #     self.slice_id = slice_id

    def __init__(self, node_masks, spacing):
        self.node_masks = node_masks;
        self.spacing = spacing;
        logger.info("DataLoader initialized")

    # Criteria 1: Minimum distance between the two nodes in this slice        
    def calculate_min_distance_single_slice(self, node_a, node_b, slice_id, debug=False):
        """
        Calculate minimum distance between two nodes in a single specified slice.
        
        Args:
            node_a: First node identifier
            node_b: Second node identifier
            slice_id: The specific slice to analyze
            
        Returns:
            Minimum distance between the two nodes in the specified slice
            and a visualization for debugging
        """
        # Get the 3D masks
        mask_a_3d = sitk.GetArrayFromImage(self.node_masks[node_a]) > 0
        mask_b_3d = sitk.GetArrayFromImage(self.node_masks[node_b]) > 0
        
        # Extract only the specified slice
        if slice_id < 0 or slice_id >= mask_a_3d.shape[0]:
            logger.error(f"Slice ID {slice_id} out of range (0-{mask_a_3d.shape[0]-1})")
            return np.inf, None
        
        # Extract the 2D masks for the specified slice
        mask_a = mask_a_3d[slice_id]
        mask_b = mask_b_3d[slice_id]
        
        # If either mask is empty in this slice, return infinity
        if not np.any(mask_a) or not np.any(mask_b):
            logger.warn(f"One or both masks are empty in slice {slice_id}")
            return np.inf, None
        
        # Get coordinates of boundary pixels
        # A pixel is on the boundary if it's part of the mask and has at least one neighbor that isn't
        struct = ndimage.generate_binary_structure(2, 1)  # 2D connectivity
        eroded_a = ndimage.binary_erosion(mask_a, struct)
        boundary_a = mask_a & ~eroded_a
        
        eroded_b = ndimage.binary_erosion(mask_b, struct)
        boundary_b = mask_b & ~eroded_b
        
        # Get indices of boundary pixels
        boundary_a_indices = np.argwhere(boundary_a)
        boundary_b_indices = np.argwhere(boundary_b)
        
        # Convert indices to physical coordinates using spacing
        # Using only the x,y components of spacing for 2D
        spacing_xy = self.spacing[0:2]
        boundary_a_coords = boundary_a_indices * spacing_xy
        boundary_b_coords = boundary_b_indices * spacing_xy
        logger.debug(f"Spacing being used: {self.spacing}")
        logger.debug(f"Spacing_xy: {spacing_xy}")
        
        # Calculate minimum distance using KDTree for efficiency
        from scipy.spatial import KDTree
        
        if len(boundary_a_coords) == 0 or len(boundary_b_coords) == 0:
            logger.warning(f"One or both boundaries are empty in slice {slice_id}")
            return np.inf, None
        
        tree_a = KDTree(boundary_a_coords)
        tree_b = KDTree(boundary_b_coords)
        
        # Find minimum distance from A to B and get the closest points
        distances_a_to_b, indices_a_to_b = tree_a.query(boundary_b_coords)
        min_dist_a_to_b = np.min(distances_a_to_b)
        min_idx_a_to_b = indices_a_to_b[np.argmin(distances_a_to_b)]
        closest_point_a = boundary_a_coords[min_idx_a_to_b]
        closest_point_b_from_a = boundary_b_coords[np.argmin(distances_a_to_b)]
        
        # Find minimum distance from B to A and get the closest points
        distances_b_to_a, indices_b_to_a = tree_b.query(boundary_a_coords)
        min_dist_b_to_a = np.min(distances_b_to_a)
        min_idx_b_to_a = indices_b_to_a[np.argmin(distances_b_to_a)]
        closest_point_b = boundary_b_coords[min_idx_b_to_a]
        closest_point_a_from_b = boundary_a_coords[np.argmin(distances_b_to_a)]
        
        # Determine which is the minimum distance
        if min_dist_a_to_b <= min_dist_b_to_a:
            min_dist = min_dist_a_to_b
            closest_pair = (closest_point_a, closest_point_b_from_a)
        else:
            min_dist = min_dist_b_to_a
            closest_pair = (closest_point_a_from_b, closest_point_b)
        
        if debug:
            # Create visualization for debugging
            visualization = self.create_distance_visualization(
                mask_a, mask_b, boundary_a, boundary_b, 
                closest_pair, min_dist, slice_id, node_a, node_b
            )
        
        return min_dist
    
    def create_distance_visualization(self, mask_a, mask_b, boundary_a, boundary_b, 
                                    closest_pair, min_dist, slice_id, node_a, node_b):
        """
        Create a visualization image for debugging the distance calculation.
        
        Args:
            mask_a, mask_b: Binary masks for the two nodes
            boundary_a, boundary_b: Binary masks for the boundaries
            closest_pair: Tuple of coordinates for the closest points
            min_dist: The calculated minimum distance
            slice_id: The slice being visualized
            node_a, node_b: Node identifiers
            
        Returns:
            A matplotlib figure object with the visualization
        """
        import matplotlib.pyplot as plt
        from matplotlib.patches import ConnectionPatch
        
        # Create a figure
        fig, ax = plt.subplots(figsize=(10, 8))
        
        # Create a combined image for visualization - RGB only (no alpha channel)
        vis_img = np.zeros((*mask_a.shape, 3), dtype=float)
        
        # Fill with original masks (using semi-transparent colors)
        vis_img[mask_a, 0] = 0.7  # Red component for mask A
        vis_img[mask_b, 2] = 0.7  # Blue component for mask B
        
        # Highlight the boundaries
        vis_img[boundary_a, 0] = 1.0  # Bright red for boundary A
        vis_img[boundary_b, 2] = 1.0  # Bright blue for boundary B
        
        # Display the image
        ax.imshow(vis_img)
        
        # Add the connection line between the closest points
        if closest_pair:
            point_a, point_b = closest_pair
            # Convert from physical coordinates back to pixel indices
            spacing_xy = self.spacing[0:2]
            idx_a = point_a / spacing_xy
            idx_b = point_b / spacing_xy
            
            # Draw a line connecting the closest points
            ax.add_patch(ConnectionPatch(
                xyA=(idx_a[1], idx_a[0]),
                xyB=(idx_b[1], idx_b[0]),
                coordsA="data", coordsB="data",
                axesA=ax, axesB=ax,
                color="yellow", linewidth=2
            ))
            
            # Mark the points
            ax.plot(idx_a[1], idx_a[0], 'o', color='green', markersize=8)
            ax.plot(idx_b[1], idx_b[0], 'o', color='green', markersize=8)
            
        # Add labels and title
        ax.set_title(f"Distance between nodes {node_a} and {node_b} in slice {slice_id}: {min_dist:.2f} units")
        ax.set_xlabel("X")
        ax.set_ylabel("Y")
        
        # Add a legend
        from matplotlib.patches import Patch
        legend_elements = [
            Patch(facecolor='red', alpha=0.5, label=f'Node {node_a}'),
            Patch(facecolor='blue', alpha=0.5, label=f'Node {node_b}'),
            Patch(facecolor='yellow', label='Minimum distance')
        ]
        ax.legend(handles=legend_elements, loc='upper right')
        
        plt.tight_layout()
        
        return fig

    def dilate_mask(self, mask: sitk.Image) -> sitk.Image:
        """
        Dilate a binary mask with SimpleITK .
        The operation is applied to every axial (x–y) slice individually.

        Parameters
        mask : sitk.Image
            3-D binary image (0 background, >0 foreground).

        Returns
        sitk.Image
            Dilated 3-D mask (uint8, 0/1) with the same meta-data as the input.
        """
        # 1. Ensure the mask is strictly 0/1
        original_mask = mask
        binary_mask   = sitk.Cast(mask > 0, sitk.sitkUInt8)

        original_count = int(sitk.GetArrayViewFromImage(binary_mask).sum())

        # 2. Prepare the 2-D extractor and dilater
        size  = list(binary_mask.GetSize()) # [x, y, z]
        depth = size[2]

        extractor = sitk.ExtractImageFilter()
        extractor.SetSize([size[0], size[1], 0])

        dilater = sitk.BinaryDilateImageFilter()
        dilater.SetForegroundValue(1)
        dilater.SetBackgroundValue(0)
        dilater.SetKernelType(sitk.sitkBall)
        dilater.SetKernelRadius(DILATION_RADIUS)

        # 3. Dilate every slice and collect the results
        dilated_slices = []
        for z in range(depth):
            extractor.SetIndex([0, 0, z])
            slice2d        = extractor.Execute(binary_mask)
            dilated_slice  = dilater.Execute(slice2d)
            dilated_slices.append(dilated_slice)

        # 4. Stack the 2-D slices back into a 3-D volume
        dilated_volume = sitk.JoinSeries(dilated_slices)
        # Restoring the original meta-data
        dilated_volume.CopyInformation(original_mask)

        # 5. Logging
        dilated_count = int(sitk.GetArrayViewFromImage(dilated_volume).sum())
        logger.info(
            f"  Dilation: {original_count} voxels -> {dilated_count} voxels "
            f"(+{dilated_count - original_count}, "
            f"{dilated_count / max(original_count, 1):.2f}x)"
        )

        return dilated_volume
    
    def find_contact_region(self, dilated_a, dilated_b, node_a, node_b, debug=False):
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

        contact_region = sitk.And(dilated_a, dilated_b)
        if debug:
            filename_ab_cont = f"{timestamp}_node_{node_a}_{node_b}_cont.nii.gz"
            sitk.WriteImage(contact_region, filename_ab_cont)
            logger.info(f"Saved {filename_ab_cont}")

        return contact_region

    
    # TODO: Criteria 2: After dilation, area of overlapping region in this slice
    def calculate_contact_area(self, contact_region_slice):
        """Note that contact_region_slice should be a single layer (2D not 3D)"""

        np_contact = sitk.GetArrayFromImage(contact_region_slice)
        spacing_xy = self.spacing[0:2]
        voxel_area = np.prod(spacing_xy)
        contact_voxels = np.sum(np_contact)
        area = contact_voxels * voxel_area

        logger.debug(f"Spacing xy: {spacing_xy}")
        logger.debug(f"Voxel area: {voxel_area}")
        logger.debug(f"Contact region: {contact_voxels} voxels")
        logger.debug(f"Contact area: {area} mm2")

        return area
        

    # TODO: Criteria 3: After dilation, intensity of overlapping region in this slice, relative to intensity of each of the two nodes     


In [ ]:
# TODO: class MajorityCounter: (?)

In [ ]:
# TODO: class AnnotationMerger

In [ ]:
from platform import node


def run_pipeline_on_case(mri_path, annotation_path, output_path, debug=False):
    dataloader = DataLoader(mri_path=mri_path, annotation_path=annotation_path)
    dataloader.load_data();

    node_labels = dataloader.node_labels

    node_pairs = [(a, b) for i, a in enumerate(node_labels) 
                     for b in node_labels[i+1:]]
        
    logger.info(f"Analyzing {len(node_pairs)} node pairs")

    node_masks = dataloader.node_masks
    spacing = dataloader.spacing
    sliceanalyzer = SliceAnalyzer(node_masks=node_masks, spacing=spacing);

    
    for node_a, node_b in node_pairs:
        logger.info(f"Analyzing node pair ({node_a}, {node_b})")
        common_list = dataloader.get_common_slices(node_a=node_a, node_b=node_b)

        if common_list:

            # Initialize variables
            num_mat_slices = 0
            len_common_list = len(common_list)
            index_list = [f"{node_a} and {node_b}"] * len_common_list
            slide_id_list = common_list
            c1_list = [False] * len_common_list
            ful_c1 = False
            c2_list = [False] * len_common_list
            ful_c2 = False
            c3_list = [False] * len_common_list
            ful_c3 = False

            logger.info(f"Analyzing node pair ({node_a}, {node_b}) since they have slides in common")
            logger.info(f"Starting analysis of criteria 1 for node pair ({node_a}, {node_b})")

            for slice_id in common_list:
                index_slice_id = common_list.index(slice_id)
                logger.info(f"Analyzing criteria 1 for node pair ({node_a}, {node_b}) in slice {slice_id}")
                logger.debug(f"Index of this slice in the common list is {index_slice_id}")

                min_dist = sliceanalyzer.calculate_min_distance_single_slice(node_a=node_a, node_b=node_b, slice_id=slice_id)
                logger.info(f"Minimum distance is {min_dist}")
                
                if min_dist < DISTANCE_THERSHOLD:
                    logger.debug(f"Minimum distance lower than threshold {DISTANCE_THERSHOLD}")
                    c1_list[index_slice_id] = True
                else: 
                    logger.debug(f"Minimum distance not lower than threshold {DISTANCE_THERSHOLD}")
            
            logger.info(f"In node pair ({node_a}, {node_b}), number of slices that fulfilled criteria 1 is {sum(c1_list)} out of {len_common_list}")

            if sum(c1_list) >= (len_common_list/2):
                logger.info(f"In node pair ({node_a}, {node_b}), half or more than half of total slices fulfilled criteria 1, proceeding to criteria 2 analysis")
                ful_c1 = True 
            else:
                logger.info(f"In node pair ({node_a}, {node_b}), less than half of total slices fulfilled criteria 1, skipping further analysis")
                 
            if ful_c1:
                logger.info(f"Starting analysis of criteria 2 for node pair ({node_a}, {node_b})")
                logger.info(f"Dilating masks of node pair ({node_a}, {node_b}) with dilation radius {DILATION_RADIUS}")

                # Get original masks
                original_a = node_masks[node_a]
                original_b = node_masks[node_b]
                
                # Dilate both masks
                dilated_a = sliceanalyzer.dilate_mask(original_a)
                dilated_b = sliceanalyzer.dilate_mask(original_b)

                if debug:
                    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

                    filename_a_orig = f"{timestamp}_node_{node_a}_original.nii.gz"
                    filename_a_dil = f"{timestamp}_node_{node_a}_dilated.nii.gz"
                    
                    sitk.WriteImage(original_a, filename_a_orig)
                    logger.info(f"Saved {filename_a_orig}")
                    sitk.WriteImage(dilated_a, filename_a_dil)
                    logger.info(f"Saved {filename_a_dil}")

                    filename_b_orig = f"{timestamp}_node_{node_b}_original.nii.gz"
                    filename_b_dil = f"{timestamp}_node_{node_b}_dilated.nii.gz"
                    
                    sitk.WriteImage(original_b, filename_b_orig)
                    logger.info(f"Saved {filename_b_orig}")
                    sitk.WriteImage(dilated_b, filename_b_dil)
                    logger.info(f"Saved {filename_b_dil}")

                for slice_id in common_list:
                    index_slice_id = common_list.index(slice_id)
                    logger.info(f"Analyzing criteria 2 for node pair ({node_a}, {node_b}) in slice {slice_id}")
                    logger.debug(f"Index of this slice in the common list is {index_slice_id}")
                    
                    contact_region = sliceanalyzer.find_contact_region(dilated_a=dilated_a, dilated_b=dilated_b, node_a=node_a, node_b=node_b, debug=DEBUG)
                    
                    size = list(contact_region.GetSize())
                    extractor = sitk.ExtractImageFilter()
                    extractor.SetSize([size[0], size[1], 0]) # Sets the size of the extracted thing
                    z = slice_id
                    extractor.SetIndex([0, 0, z]) # Sets where to start extracting from
                    contact_region_slice = extractor.Execute(contact_region)

                    contact_area = sliceanalyzer.calculate_contact_area(contact_region_slice=contact_region_slice)

                    logger.info(f"Contact area of ({node_a}, {node_b}) in slice {slice_id} is {contact_area} mm2")                   

In [ ]:
if __name__ == "__main__":
    mri_path = "data/raw/images/1077-T2_FS_TRA+301.nii.gz"
    annotation_path = "data/raw/labels/1077-T2_FS_TRA+301.nii.gz"
    output_path = "test.nii.gz"

    run_pipeline_on_case(mri_path=mri_path, annotation_path=annotation_path, output_path=output_path, debug=DEBUG)

    # TODO: scale up to folder instead of a single file